# ATH operational-v2: Qwen3.5 9B follow-up on Colab

This notebook tries `qwen3.5:9b` after the 4B run failed to complete its investigations. Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. Ollama lists the 9B Q4_K_M model at approximately 6.6 GB; GPU residency is checked before evaluation.

The evaluator source, prompts, three development cases, six evaluation cases, sampling, two-probe limit, 120-second investigation deadline, and 1,536-token output allowance stay fixed. A larger model may still fail these constraints. Both splits get new freezes and paired deterministic baselines. Outputs and downloads use a separate 9B name.

**Exploratory follow-up:** these six evaluation cases were already inspected in the 4B review. The evaluator retains the historical split name `heldout`, but this run is not fresh held-out validation. The existing success rule also has an evidence-recovery ceiling: the baseline already recovers all useful events. Read individual metrics, not only the conclusion label.

The model is preloaded without a task prompt, outside measured investigation time. This differs from the earlier notebook; do not attribute a cold-start latency improvement solely to model capability. All three development model rows must complete before the notebook starts the larger evaluation.

Sources: [Ollama model](https://ollama.com/library/qwen3.5:9b), [Qwen model card](https://huggingface.co/Qwen/Qwen3.5-9B).


## 0. Configuration
The source digest must match the reviewed 4B evaluator. A changed source requires a separate experiment. Ollama is pinned; the model digest and configuration are recorded in each freeze. Use a fresh T4 session and do not restore the old 4B archives here.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shayb1187-a11y/agent-threat-hunter.git"
BRANCH = "m14-real-data-validation"
MINIMUM_COMMIT = "f48379cd173d1587bc54f820ccb0885de2636b43"
MODEL = "qwen3.5:9b"
RUN_ID = "qwen35-9b-followup"
EXPECTED_SOURCE_SHA256 = "f5efc927405f02a44a41ca523dd5c34beeb20405f155f4847f241b8d4ddcdaf6"
OLLAMA_VERSION = "0.34.1"
REPO = Path("/content/agentic-threat-hunter")
OUTPUT = Path("/content") / f"ath-auth-execution-{RUN_ID}"
DEV = OUTPUT / "dev"
HELDOUT = OUTPUT / "heldout"
print({"model": MODEL, "output": str(OUTPUT)})

## 1. Verify the GPU
Use a fresh T4 GPU session. This check confirms the device is available; the model preload later checks actual residency. A model that spills onto CPU can miss the unchanged 120-second investigation deadline.


In [ ]:
import shutil, subprocess

assert shutil.which("nvidia-smi"), "No NVIDIA GPU: Runtime → Change runtime type → T4 GPU"
subprocess.run(["nvidia-smi"], check=True)

## 2. Clone the public branch and install ATH


In [ ]:
import os, sys

git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=git_env)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True, env=git_env)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True, env=git_env)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True, env=git_env)
subprocess.run(["git", "merge-base", "--is-ancestor", MINIMUM_COMMIT, "HEAD"], cwd=REPO, check=True)
subprocess.run(["git", "diff", "--quiet"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)
os.chdir(REPO)
print("commit", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("python", sys.version.split()[0])
source_digest = subprocess.check_output([sys.executable, "-c", "from ath.evaluation.auth_execution import source_hash; print(source_hash())"], cwd=REPO, text=True).strip()
assert source_digest == EXPECTED_SOURCE_SHA256, "Evaluator source differs from the reviewed 4B run; use its source revision for this model-only follow-up."


## 3. Install Ollama, start one inference slot, and pull the frozen model


In [ ]:
import json, time, urllib.request

def daemon_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not shutil.which("ollama"):
    install = f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh"
    subprocess.run(install, shell=True, check=True)
if not daemon_up():
    log = open("/content/ollama-auth-execution.log", "ab")
    daemon_env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_KEEP_ALIVE": "-1"}
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=daemon_env)
    for _ in range(120):
        if daemon_up(): break
        time.sleep(1)
    else: raise RuntimeError("Ollama did not start; inspect /content/ollama-auth-execution.log")
version = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/version"))["version"]
assert version == OLLAMA_VERSION, (version, OLLAMA_VERSION)
subprocess.run(["ollama", "pull", MODEL], check=True)
print("Ollama", version, "with", MODEL)

## 4. Optional: restore this model's checkpoint
Skip on a fresh session. Only use a checkpoint exported by this 9B follow-up. The restore checks paths and refuses conflicting existing files. Original 4B archives belong to their original output directory.


In [ ]:
# OPTIONAL: uncomment only to resume this 9B follow-up.
# from google.colab import files
# import io, zipfile
# uploaded = files.upload()
# for name, blob in uploaded.items():
#     with zipfile.ZipFile(io.BytesIO(blob)) as archive:
#         pending = []
#         for member in archive.infolist():
#             target = (Path('/content') / member.filename).resolve()
#             assert OUTPUT.resolve() in target.parents, 'Wrong model checkpoint or invalid path'
#             if member.is_dir(): continue
#             content = archive.read(member)
#             if target.exists():
#                 assert target.read_bytes() == content, f'Conflicting existing file: {target}'
#             else:
#                 pending.append((target, content))
#         for target, content in pending:
#             target.parent.mkdir(parents=True, exist_ok=True)
#             with target.open('xb') as handle: handle.write(content)
# print('restored', sorted(uploaded))


## 5. Freeze development and held-out protocols before inference
Both freezes are created before any model call. Do not edit source, configuration, prompts, cases, repeats, or decision criteria after this cell. If a freeze already exists, this cell validates it through `summarise` rather than replacing it.


In [ ]:
def ath(*args, allow=(0,)):
    command = [sys.executable, "-m", "ath.evaluation.auth_execution", *map(str, args)]
    print("+", " ".join(command), flush=True)
    result = subprocess.run(command, cwd=REPO)
    if result.returncode not in allow:
        raise RuntimeError(f"command exited {result.returncode}")
    return result.returncode

OUTPUT.mkdir(parents=True, exist_ok=True)
for path, split, repeats in ((DEV, "dev", 1), (HELDOUT, "heldout", 2)):
    if not (path / "FREEZE.json").exists():
        ath("freeze", "--out", path, "--split", split, "--repeats", repeats, "--model", MODEL)
    else:
        existing = json.loads((path / "FREEZE.json").read_text())
        assert existing["model_configuration"]["model"] == MODEL, "Different model: use a new output directory"
        assert (existing["split"], existing["repeats"]) == (split, repeats), "Freeze layout differs"
        ath("summarise", "--out", path)
print("Both experiment identities are frozen.")
context = {"run_id": RUN_ID, "model": MODEL,
           "study": "exploratory model-size follow-up on previously inspected cases",
           "fresh_holdout": False, "source_sha256": source_digest,
           "preload_before_cases": True, "investigation_budgets_changed": False}
context_path = OUTPUT / "RUN_CONTEXT.json"
if context_path.exists():
    assert json.loads(context_path.read_text()) == context, "Run context differs; use a new directory"
else:
    with context_path.open("x") as handle: json.dump(context, handle, indent=2)


## 6. Run both deterministic baselines in this runtime
These rows are fast and provide the paired Colab baseline.


In [ ]:
ath("run", "--out", DEV, "--arm", "deterministic")
ath("run", "--out", HELDOUT, "--arm", "deterministic")

## 7. Preload the model and run development
An empty request loads model weights without an investigation prompt. Loading is recorded separately from case latency. All three D1 rows must complete to enable the next split; otherwise download the checkpoint and inspect failures. Saved failed rows are preserved, not retried in place.


In [ ]:
from datetime import datetime, timezone
from ath.evaluation.auth_execution import _client

client = _client(MODEL)
request = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps({"model": MODEL, "stream": False, "keep_alive": -1,
                     "options": {"num_ctx": client.num_ctx}}).encode(),
    headers={"Content-Type": "application/json"},
)
started = time.perf_counter()
with urllib.request.urlopen(request, timeout=300) as response:
    preload_response = json.load(response)
assert not preload_response.get("error"), preload_response
residency = client.residency()
preload = {"model": MODEL, "seconds": time.perf_counter() - started,
           "residency": residency, "task_prompt_sent": False,
           "outside_investigation_timing": True}
preloads = OUTPUT / "preloads"
preloads.mkdir(exist_ok=True)
with (preloads / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")).open("x") as handle:
    json.dump(preload, handle, indent=2)
print(json.dumps(preload, indent=2))
subprocess.run(["ollama", "ps"], check=True)
assert residency["size"] and residency["size_vram"] >= residency["size"], "Model is not fully GPU resident. Use a fresh T4 session before starting cases."

ath("run", "--out", DEV, "--arm", "d1", allow=(0, 3))
ath("summarise", "--out", DEV)
summary = json.loads((DEV / "SUMMARY.json").read_text())
print(json.dumps(summary, indent=2))
dev_ready = (summary["complete_comparison"] and summary["arms"]["d1"]["complete"] == summary["arms"]["d1"]["rows"] == 3)
print("Development executions passed; evaluation enabled." if dev_ready else "STOP after downloading the development checkpoint: one or more investigations failed or are missing.")
for path in sorted((DEV / "rows").glob("*_d1_*.json")):
    row = json.loads(path.read_text())
    if not row["scores"]["complete"]:
        print(path.name, row["state"]["investigation"]["operational"]["reasons"])


## 8. Download a development checkpoint


In [ ]:
import zipfile
from google.colab import files

def export_results(name):
    target = Path('/content') / name
    if target.exists(): target.unlink()
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file(): archive.write(path, path.relative_to('/content'))
    print(target, target.stat().st_size, 'bytes')
    files.download(str(target))

export_results(f'ath_auth_execution_{RUN_ID}_dev_checkpoint.zip')

## 9. Run the six previously inspected evaluation cases
This cell checks saved development outcomes before any evaluation model call. It produces 12 D1 rows only if all three development investigations completed. These are exploratory repeat measurements on previously inspected cases, not fresh held-out validation. Failures remain in the denominator. Do not tune or delete saved rows.


In [ ]:
development = json.loads((DEV / "SUMMARY.json").read_text())
dev_ready = (development["complete_comparison"] and development["arms"]["d1"]["complete"] == development["arms"]["d1"]["rows"] == 3)
assert dev_ready, "Development investigations did not all complete. The checkpoint was downloaded in step 8; inspect it before another experiment."
ath("run", "--out", HELDOUT, "--arm", "d1", allow=(0, 3))
ath("summarise", "--out", HELDOUT)
summary = json.loads((HELDOUT / "SUMMARY.json").read_text())
print(json.dumps(summary, indent=2))
print("All result rows recorded:", summary["complete_comparison"])
print("Successful D1 executions:", summary["arms"]["d1"]["complete"], "/", 12)
print("Missing rows:", summary["missing_rows"])


## 10. Export the evidence bundle
Always save this archive, even if rows are missing or executions failed. It includes freezes, raw rows, reports, summaries, run context and preload records. A filename ending in `results` does not imply success. Resume missing rows under the same freeze; existing failures remain recorded. Keep the old 4B and Windows results separate.


In [ ]:
export_results(f'ath_auth_execution_{RUN_ID}_results.zip')


## Interpretation
The first question is whether the larger model completes investigations and follows the evidence contract under the original constraints. Compare completion, per-case decisions, useful evidence, verification rejections, time and tokens. Neither zero false accusations caused by forced abstention nor `complete_comparison: true` establishes success.

The 4B run has already exposed these evaluation cases. This follow-up cannot establish unseen generalization, and the unchanged evidence-recovery criterion cannot pass when the baseline already recovers all available useful events. Any redesigned protocol needs a separate freeze and new evaluation data. No model capability improvement or analyst time savings is assumed in advance.
